# HBICP iteration 3 on Colab
Runtime > Change runtime type > GPU (T4). Run the cells in order. If Colab disconnects, reopen and run all again: it resumes from `last.pt` on Drive.

Runs: `n8_ref` (reference, already done), `n8_resid` (residual anchor: o_T + dec(z_k) - dec(z_0)), `n8_resid_all` (residual + variable horizon H in [4, 32] + per-k-normalised loss), then a train-vs-test k* diagnostic (model/dataset too small?) and the fair HL baseline (`hl_fair`, CPU only, slow). The quadratic readout is off.

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
REPO = 'https://github.com/Alessio-Pecilli/Hamiltonian-Blind-In-Context-Propagator.git'
if not os.path.exists('/content/hbicp'):
    subprocess.run(['git', 'clone', REPO, '/content/hbicp'], check=True)
else:
    subprocess.run(['git', '-C', '/content/hbicp', 'pull'], check=True)
%cd /content/hbicp/Hamiltonian-Blind-In-Context-Propagator
!pip install -q h5py pyyaml scipy
# datasets, checkpoints and results live on Drive so they survive disconnects
D = '/content/drive/MyDrive/hbicp'
for d in ['datasets', 'runs', 'results']:
    os.makedirs(f'{D}/{d}', exist_ok=True)
    if not os.path.islink(d): os.symlink(f'{D}/{d}', d)
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/hbicp/Hamiltonian-Blind-In-Context-Propagator
True Tesla T4


In [ ]:
# new code paths (residual anchor, per-k loss) must pass before spending GPU time
!python -m pytest -q tests

In [5]:
# datasets (skipped if already on Drive)
!test -f datasets/n8/test.h5 || bash scripts/make_datasets.sh local 2

In [6]:
# reference, residual anchor, residual + variable horizon + per-k-normalised loss; idempotent
import os, subprocess, torch
steps = 8000
runs = [('n8_ref', []), ('n8_resid', ['model.residual=true']),
        ('n8_resid_all', ['model.residual=true', 'train.var_horizon=true', 'train.H=32', 'train.H_min=4', 'loss.per_k_norm=true'])]
def step(d):
    p = f'{d}/last.pt'
    return torch.load(p, map_location='cpu', weights_only=False)['step'] if os.path.exists(p) else 0
for name, extra in runs:
    d, res = f'runs/{name}', f'results/{name}/results.json'
    if os.path.exists(res): continue
    os.makedirs(d, exist_ok=True)
    if step(d) < steps:
        a = ['python', 'train.py', '--preset', 'local', '--set', f'train.steps={steps}', 'train.seed=0', f'train.out_dir={d}'] + extra
        if os.path.exists(f'{d}/last.pt'): a += ['--resume', f'{d}/last.pt']
        subprocess.run(a, check=True)
    if step(d) >= steps:
        subprocess.run(['python', '-m', 'eval.evaluate', '--ckpt', f'{d}/last.pt', '--data', 'datasets/n8/test.h5',
                        '--T', '8', '16', '32', '64', '--horizon', '64',
                        '--baselines', 'persistence', 'dmd', 'dmd4', 'oracle', 'hl_oracle', '--out', f'results/{name}'], check=True)

In [ ]:
# k* on training vs test windows: train >> test = dataset too small (overfit); train ~ test and both low = model undertrained / too small
import json, subprocess
for name in ['n8_ref', 'n8_resid', 'n8_resid_all']:
    if not os.path.exists(f'runs/{name}/last.pt'): continue
    out = f'results/{name}_traintest'
    for split in ['train', 'test']:
        if not os.path.exists(f'{out}/{split}/results.json'):
            subprocess.run(['python', '-m', 'eval.evaluate', '--ckpt', f'runs/{name}/last.pt', '--data', f'datasets/n8/{split}.h5',
                            '--T', '8', '16', '32', '64', '--horizon', '64', '--baselines', '--max_traj', '64', '--out', f'{out}/{split}'], check=True)
    for split in ['train', 'test']:
        r = next(iter(json.load(open(f'{out}/{split}/results.json')).values()))

## Fair HL baseline
Generic 2-local Hamiltonian fitted on the context only (`hl_fair_o0`: psi_0 from o_0; `hl_fair_fit`: psi_0 fitted). CPU, roughly 1 min per trajectory per T at n=8: lower `N_TRAJ` for a quick pass. Independent of the models; result goes to `results/hl_fair` and is added to the quad publish.

In [ ]:
# fair HL baseline on the product test set (skipped if already on Drive)
import os, subprocess
N_TRAJ = 20
if not os.path.exists('results/hl_fair/results.json'):
    subprocess.run(['python', '-m', 'eval.evaluate', '--data', 'datasets/n8/test.h5', '--T', '16', '32', '64', '--horizon', '64',
                    '--baselines', 'hl_fair_o0', 'hl_fair_fit', '--max_traj', str(N_TRAJ), '--out', 'results/hl_fair'], check=True)

## Publish
One-time setup: GitHub > Settings > Developer settings > Fine-grained tokens > only this repo, Contents: Read and write. In Colab, left sidebar key icon > Add secret `GH_TOKEN` with the token, enable Notebook access.

In [7]:
# publish to the dashboard (git push) from Colab; needs Colab secret GH_TOKEN (see markdown above)
import os, shutil, subprocess
from google.colab import userdata
tok = userdata.get('GH_TOKEN')
subprocess.run(['git', 'remote', 'set-url', 'origin', f'https://{tok}@github.com/Alessio-Pecilli/Hamiltonian-Blind-In-Context-Propagator.git'], check=True)
subprocess.run(['git', 'config', 'user.name', 'Alessio Pecilli'], check=True)
subprocess.run(['git', 'config', 'user.email', 'alessiopecilli@gmail.com'], check=True)
jobs = [('n8_ref', ['n8_ref'], 'n8 A riferimento (no probe, no opzioni coppie, S off)',
         'Run A: riferimento. pair_attn/triangle/pair_latent spenti, nessun L_probe, use_S=false, n=8, 8000 step, seed 0, validazione ogni 1000 step.'),
        ('n8_resid', ['n8_ref', 'n8_resid'], 'n8 residual anchor: o_T + dec(z_k) - dec(z_0)',
         'Run residual: come A ma model.residual=true. La predizione e ancorata all ultimo frame osservato e il latente modella solo la variazione: errore -> 0 per k -> 0 come DMD. Stesso seed e stessi step di A.'),
        ('n8_resid_all', ['n8_ref', 'n8_resid', 'n8_resid_all', 'hl_fair'], 'n8 residual + orizzonte variabile + loss normalizzata per k (+ HL equo)',
         'Run resid_all: residual anchor + orizzonte variabile (H uniforme in [4, 32] a ogni step) + loss normalizzata per k (ogni k diviso per l errore della persistenza a quel k). Readout quadratico spento. Include la baseline HL equa se calcolata.')]
for name, keep, label, notes in jobs:
    marker = f'results/{name}/.published'
    if os.path.exists(marker): print(name, 'skip: gia pubblicato (.published)'); continue
    if not os.path.exists(f'results/{name}/results.json'): print(name, 'skip: manca results/%s/results.json' % name); continue
    shutil.rmtree('stage_publish', ignore_errors=True)
    for k in keep:
        if not os.path.exists(f'results/{k}/results.json'): continue  # e.g. hl_fair not computed yet
        os.makedirs(f'stage_publish/runs/{k}', exist_ok=True); os.makedirs(f'stage_publish/results/{k}')
        if k != 'hl_fair':
            for f in ['config.json', 'log.jsonl', 'last.pt']: shutil.copy(f'runs/{k}/{f}', f'stage_publish/runs/{k}')
        shutil.copy(f'results/{k}/results.json', f'stage_publish/results/{k}')
    subprocess.run(['git', 'pull', '--rebase', '-q'], check=True)
    r = subprocess.run(['python', '-m', 'tools.publish_dashboard', '--label', label, '--notes', notes, '--runs', 'stage_publish/runs',
        '--results', 'stage_publish/results', '--data', 'datasets/n8/test.h5', '--ckpt', f'runs/{name}/last.pt',
        '--T', '32', '--horizon', '64', '--sample', '0', '--keep', '100'])
    if r.returncode == 0: open(marker, 'w').close()
    shutil.rmtree('stage_publish', ignore_errors=True)
print('done')
